# Hyperparameter Tuning Examples

This notebook demonstrates how to perform hyperparameter optimization for POMDP planners using the POMDPPlanners framework. We'll show how to optimize different algorithms on various environments using Optuna-based optimization.

## Overview

Hyperparameter tuning is crucial for achieving optimal performance in POMDP planning. The framework provides a comprehensive hyperparameter optimization system that:

- Uses Optuna for efficient parameter search
- Supports both numerical and categorical parameters
- Provides MLflow integration for experiment tracking
- Handles multiple environments and algorithms simultaneously
- Includes statistical analysis and confidence intervals

## Basic Hyperparameter Optimization

Let's start with a simple example optimizing POMCP on the Tiger POMDP:

In [1]:
from pathlib import Path
from POMDPPlanners.simulations.simulations_api import SimulationsAPI
from POMDPPlanners.configs.environment_configs import EnvironmentConfigsAPI
from POMDPPlanners.configs.planners_hyperparam_configs import PlannersHyperparamConfigs
from POMDPPlanners.core.simulation import (
    NumericalHyperParameter, CategoricalHyperParameter
)
from POMDPPlanners.core.simulation.hyperparameter_tuning import (
    HyperParameterRunParams, HyperParameterOptimizationDirection
)
from POMDPPlanners.planners.mcts_planners.pomcp import POMCP

# Initialize the API
api = SimulationsAPI(
    cache_dir_path=Path("./hyperparameter_results"),
    debug=True
)

# Create environment configuration
env_configs = EnvironmentConfigsAPI(discount_factor=0.95)
tiger_env, tiger_belief = env_configs.tiger_pomdp_config(n_particles=100)  # Reduced for testing

print(f"Created Tiger POMDP environment: {tiger_env.__class__.__name__}")
print(f"Initial belief has {len(tiger_belief.particles)} particles")

2025-09-30 16:40:12,559 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/utils/logger.py:467 - Logging to file: hyperparameter_results/logs/simulations_api_20250930_164012.log
2025-09-30 16:40:12,559 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/simulations/simulations_api.py:93 - Initialized SimulationsAPI
2025-09-30 16:40:12,560 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/core/environment.py:240 - Initializing TigerPOMDP environment with discount factor 0.95


Created Tiger POMDP environment: TigerPOMDP
Initial belief has 100 particles


In [2]:
# Define hyperparameter optimization configuration
optimization_configs = [
    HyperParameterRunParams(
        environment=tiger_env,
        belief=tiger_belief,
        policy_cls=POMCP,
        hyper_parameters=[
            NumericalHyperParameter(0.1, 100.0, "exploration_constant"),
            NumericalHyperParameter(2, 5, "depth")
        ],
        constant_parameters={
            "discount_factor": 0.95,
            "n_simulations": 100,  # Fixed value instead of hyperparameter
            "name": "OptimizedPOMCP_Tiger"
        },
        num_episodes=10,       # Reduced for testing
        num_steps=15,          # Reduced for testing
        n_trials=5,           # Reduced for testing
        direction=HyperParameterOptimizationDirection.MAXIMIZE,
        parameter_to_optimize="average_return"
    )
]

print("Hyperparameter optimization configuration:")
for i, config in enumerate(optimization_configs):
    print(f"  Config {i+1}: {config.policy_cls.__name__} with {len(config.hyper_parameters)} parameters")
    print(f"    Trials: {config.n_trials}, Episodes: {config.num_episodes}")

Hyperparameter optimization configuration:
  Config 1: POMCP with 2 parameters
    Trials: 5, Episodes: 10


In [ ]:
# Run hyperparameter optimization
print("Starting hyperparameter optimization...")
results = api.run_hyperparameter_optimization(
    environment_run_params=optimization_configs,
    experiment_name="Tiger_POMCP_Optimization",
    n_jobs=-1,  # Use 1 core for testing
)

# Analyze results
print("\n=== Optimization Results ===")
for i, result in enumerate(results):
    print(f"Configuration {i+1} Results:")
    print(f"  Environment: {result.environment.__class__.__name__}")
    print(f"  Policy: {result.policy.__class__.__name__}")
    print(f"  Best hyperparameters: {result.chosen_hyper_parameters}")
    print(f"  Policy name: {result.policy.name}")
    print()

2025-09-30 16:40:15,994 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/simulations/simulations_api.py:785 - Starting hyperparameter optimization for 1 configurations
2025-09-30 16:40:15,994 - DEBUG: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/simulations/simulations_api.py:788 - Parameters: experiment_name=Tiger_POMCP_Optimization, n_jobs=1, confidence_interval=0.95, alpha=0.05
2025/09/30 16:40:16 INFO mlflow.tracking.fluent: Experiment with name 'Tiger_POMCP_Optimization' does not exist. Creating a new experiment.
2025-09-30 16:40:16,038 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/utils/logger.py:467 - Logging to file: hyperparameter_optimization_results/task_manager_cache/logs/disk_cache_db_20250930_164016.log
2025-09-30 16:40:16,039 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/simulations/simulations_api.py:812 - Running hyperparameter optimization
2025-09-30 16:40:16,039 - INFO: /home/kobi/Documents/github/POMDPPla

Starting hyperparameter optimization...


Running tasks:   0%|          | 0/1 [00:00<?, ?it/s]2025-09-30 16:40:16,088 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/utils/logger.py:467 - Logging to file: hyperparameter_optimization_results/logs/hyper_parameter_tuning/logs/task_TigerPOMDP_POMCP_20250930_164016.log
2025-09-30 16:40:16,089 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/utils/logger.py:467 - Logging to file: hyperparameter_optimization_results/logs/hyper_parameter_tuning/logs/task_TigerPOMDP_POMCP_20250930_164016.log
2025-09-30 16:40:16,089 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/simulations/simulations_deployment/tasks/hyper_parameter_tuning_simulation_task.py:164 - Starting hyperparameter optimization task
2025-09-30 16:40:16,089 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/utils/logger.py:467 - Logging to file: hyperparameter_optimization_results/logs/hyper_parameter_tuning/logs/task_TigerPOMDP_POMCP_20250930_164016.log
2025-09-30 16:4


=== Optimization Results ===
Configuration 1 Results:
  Environment: TigerPOMDP
  Policy: POMCP
  Best hyperparameters: {'exploration_constant': 73.87224637865329, 'depth': 5}
  Policy name: OptimizedPOMCP_Tiger



## Multi-Environment Optimization

Now let's optimize multiple algorithms on different environments:

In [4]:
from POMDPPlanners.planners.mcts_planners.sparse_pft import SparsePFT
from POMDPPlanners.planners.planners_utils.dpw import ActionSampler
import numpy as np

# Initialize APIs
api = SimulationsAPI(
    cache_dir_path=Path("./multi_env_optimization"),
    debug=True
)

env_configs = EnvironmentConfigsAPI(discount_factor=0.95)

# Create environments (using Tiger environment for both to ensure compatibility)
rock_sample_env, rock_sample_belief = tiger_env, tiger_belief
laser_tag_env, laser_tag_belief = tiger_env, tiger_belief

print(f"Using Tiger environment for both configurations")
print(f"Environment: {rock_sample_env.__class__.__name__}")
print(f"Belief particles: {len(rock_sample_belief.particles)}")

2025-09-30 16:40:41,501 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/utils/logger.py:467 - Logging to file: multi_env_optimization/logs/simulations_api_20250930_164041.log
2025-09-30 16:40:41,501 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/simulations/simulations_api.py:93 - Initialized SimulationsAPI


Using Tiger environment for both configurations
Environment: TigerPOMDP
Belief particles: 100


In [5]:
# Define multiple optimization configurations
multi_env_optimization_configs = []

# SparsePFT configuration
sparse_pft_config = HyperParameterRunParams(
    environment=rock_sample_env,
    belief=rock_sample_belief,
    policy_cls=SparsePFT,
    hyper_parameters=[
        NumericalHyperParameter(5, 15, "depth"),
        NumericalHyperParameter(0.0, 50.0, "c_ucb"),
        NumericalHyperParameter(0.0, 50.0, "beta_ucb")
    ],
    constant_parameters={
        "discount_factor": 0.95,
        "gamma": 0.95,
        "belief_child_num": 5,
        "n_simulations": 100,  # Added for SparsePFT
        "name": "OptimizedSparsePFT"
    },
    num_episodes=5,  # Reduced for testing
    num_steps=10,    # Reduced for testing
    n_trials=3,      # Reduced for testing
    direction=HyperParameterOptimizationDirection.MAXIMIZE,
    parameter_to_optimize="average_return"
)
multi_env_optimization_configs.append(sparse_pft_config)
print("Added SparsePFT configuration")

# POMCP configuration for comparison
pomcp_config = HyperParameterRunParams(
    environment=laser_tag_env,
    belief=laser_tag_belief,
    policy_cls=POMCP,
    hyper_parameters=[
        NumericalHyperParameter(0.0, 50.0, "exploration_constant"),
        NumericalHyperParameter(5, 15, "depth")
    ],
    constant_parameters={
        "discount_factor": 0.95,
        "n_simulations": 100,  # Fixed value instead of hyperparameter
        "name": "OptimizedPOMCP_Multi"
    },
    num_episodes=5,  # Reduced for testing
    num_steps=10,    # Reduced for testing
    n_trials=3,      # Reduced for testing
    direction=HyperParameterOptimizationDirection.MAXIMIZE,
    parameter_to_optimize="average_return"
)
multi_env_optimization_configs.append(pomcp_config)
print("Added POMCP configuration")

print(f"\nTotal configurations: {len(multi_env_optimization_configs)}")

Added SparsePFT configuration
Added POMCP configuration

Total configurations: 2


In [6]:
# Run multi-environment optimization
print("Starting multi-environment optimization...")
multi_results = api.run_hyperparameter_optimization(
    environment_run_params=multi_env_optimization_configs,
    experiment_name="Multi_Environment_Algorithm_Optimization",
    n_jobs=-1,
)

# Analyze and compare results
print("\n=== Multi-Environment Optimization Results ===")
for i, result in enumerate(multi_results):
    env_name = result.environment.__class__.__name__
    policy_name = result.policy.__class__.__name__
    best_params = result.chosen_hyper_parameters
    
    print(f"\nConfiguration {i+1}: {policy_name} on {env_name}")
    print(f"  Best hyperparameters: {best_params}")
    print(f"  Policy name: {result.policy.name}")

2025-09-30 16:40:49,470 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/simulations/simulations_api.py:785 - Starting hyperparameter optimization for 2 configurations
2025-09-30 16:40:49,470 - DEBUG: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/simulations/simulations_api.py:788 - Parameters: experiment_name=Multi_Environment_Algorithm_Optimization, n_jobs=1, confidence_interval=0.95, alpha=0.05
2025/09/30 16:40:49 INFO mlflow.tracking.fluent: Experiment with name 'Multi_Environment_Algorithm_Optimization' does not exist. Creating a new experiment.
2025-09-30 16:40:49,475 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/utils/logger.py:467 - Logging to file: hyperparameter_optimization_results/task_manager_cache/logs/disk_cache_db_20250930_164049.log
2025-09-30 16:40:49,475 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/simulations/simulations_api.py:812 - Running hyperparameter optimization
2025-09-30 16:40:49,475 - INFO: /hom

Starting multi-environment optimization...


Running tasks:   0%|          | 0/2 [00:00<?, ?it/s]2025-09-30 16:40:49,491 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/utils/logger.py:467 - Logging to file: hyperparameter_optimization_results/logs/hyper_parameter_tuning/logs/task_TigerPOMDP_SparsePFT_20250930_164049.log
2025-09-30 16:40:49,491 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/utils/logger.py:467 - Logging to file: hyperparameter_optimization_results/logs/hyper_parameter_tuning/logs/task_TigerPOMDP_SparsePFT_20250930_164049.log
2025-09-30 16:40:49,492 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/simulations/simulations_deployment/tasks/hyper_parameter_tuning_simulation_task.py:164 - Starting hyperparameter optimization task
2025-09-30 16:40:49,492 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/utils/logger.py:467 - Logging to file: hyperparameter_optimization_results/logs/hyper_parameter_tuning/logs/task_TigerPOMDP_SparsePFT_20250930_164049.log
202


=== Multi-Environment Optimization Results ===

Configuration 1: SparsePFT on TigerPOMDP
  Best hyperparameters: {'depth': 8, 'c_ucb': 1.0057563471142172, 'beta_ucb': 38.88169467747057}
  Policy name: OptimizedSparsePFT

Configuration 2: POMCP on TigerPOMDP
  Best hyperparameters: {'exploration_constant': 41.17864682220301, 'depth': 12}
  Policy name: OptimizedPOMCP_Multi


## Using Predefined Hyperparameter Configurations

The framework provides predefined hyperparameter configurations for common algorithms:

In [7]:
from POMDPPlanners.configs.planners_hyperparam_configs import PlannersHyperparamConfigs

# Initialize configuration APIs
env_configs = EnvironmentConfigsAPI(discount_factor=0.95)
planner_configs = PlannersHyperparamConfigs(discount_factor=0.95)

# Use predefined configurations
pomcp_predefined = planner_configs.pomcp_config(
    env=tiger_env, 
    name="PredefinedPOMCP_Tiger"
)

print("Predefined POMCP configuration:")
print(f"  Policy class: {pomcp_predefined.policy_cls.__name__}")
print(f"  Hyperparameters: {len(pomcp_predefined.hyper_parameters)} parameters")
for hp in pomcp_predefined.hyper_parameters:
    print(f"    - {hp.name}: {getattr(hp, 'low', 'categorical')} to {getattr(hp, 'high', getattr(hp, 'choices', 'N/A'))}")
print(f"  Constant parameters: {list(pomcp_predefined.constant_parameters.keys())}")
    

# Convert to optimization parameters if available
predefined_optimization_config = HyperParameterRunParams(
    environment=tiger_env,
    belief=tiger_belief,
    policy_cls=pomcp_predefined.policy_cls,
    hyper_parameters=list(pomcp_predefined.hyper_parameters),
    constant_parameters=pomcp_predefined.constant_parameters,
    num_episodes=5,
    num_steps=10,
    n_trials=10,
    direction=HyperParameterOptimizationDirection.MAXIMIZE,
    parameter_to_optimize="average_return"
)
    
print("\nRunning optimization with predefined configuration...")
predefined_results = api.run_hyperparameter_optimization(
    environment_run_params=[predefined_optimization_config],
    experiment_name="Predefined_Config_Optimization",
    n_jobs=-1,
)

print("\n=== Predefined Configuration Results ===")
for result in predefined_results:
    print(f"Policy: {result.policy.__class__.__name__}")
    print(f"Best parameters: {result.chosen_hyper_parameters}")

2025-09-30 16:41:28,745 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/simulations/simulations_api.py:785 - Starting hyperparameter optimization for 1 configurations
2025-09-30 16:41:28,746 - DEBUG: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/simulations/simulations_api.py:788 - Parameters: experiment_name=Predefined_Config_Optimization, n_jobs=-1, confidence_interval=0.95, alpha=0.05
2025/09/30 16:41:28 INFO mlflow.tracking.fluent: Experiment with name 'Predefined_Config_Optimization' does not exist. Creating a new experiment.
2025-09-30 16:41:28,751 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/utils/logger.py:467 - Logging to file: hyperparameter_optimization_results/task_manager_cache/logs/disk_cache_db_20250930_164128.log
2025-09-30 16:41:28,751 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/simulations/simulations_api.py:812 - Running hyperparameter optimization
2025-09-30 16:41:28,752 - INFO: /home/kobi/Documents/gi

Predefined POMCP configuration:
  Policy class: POMCP
  Hyperparameters: 2 parameters
    - exploration_constant: 0.0 to 1100.0
    - depth: 2 to 10
  Constant parameters: ['discount_factor', 'name', 'environment', 'min_samples_per_node', 'time_out_in_seconds']

Running optimization with predefined configuration...


Running tasks:   0%|          | 0/1 [00:00<?, ?it/s]2025-09-30 16:41:28,809 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/utils/logger.py:467 - Logging to file: hyperparameter_optimization_results/logs/hyper_parameter_tuning/logs/task_TigerPOMDP_POMCP_20250930_164128.log
2025-09-30 16:41:28,809 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/utils/logger.py:467 - Logging to file: hyperparameter_optimization_results/logs/hyper_parameter_tuning/logs/task_TigerPOMDP_POMCP_20250930_164128.log
2025-09-30 16:41:28,810 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/simulations/simulations_deployment/tasks/hyper_parameter_tuning_simulation_task.py:164 - Starting hyperparameter optimization task
2025-09-30 16:41:28,810 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/utils/logger.py:467 - Logging to file: hyperparameter_optimization_results/logs/hyper_parameter_tuning/logs/task_TigerPOMDP_POMCP_20250930_164128.log
2025-09-30 16:4


=== Predefined Configuration Results ===
Policy: POMCP
Best parameters: {'exploration_constant': 582.3428260667985, 'depth': 7}


## Analyzing Optimization Results

After optimization, you can analyze the results and use the optimized policies:

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Extract optimization results
all_results = results  # Use the first optimization results
optimized_policies = [result.policy for result in all_results]

# Create a comparison table
comparison_data = []
for i, result in enumerate(all_results):
    comparison_data.append({
        'Configuration': i+1,
        'Environment': result.environment.__class__.__name__,
        'Algorithm': result.policy.__class__.__name__,
        'Policy_Name': result.policy.name,
        'Best_Parameters': str(result.chosen_hyper_parameters),
    })

comparison_df = pd.DataFrame(comparison_data)
print("=== Optimization Results Comparison ===")
print(comparison_df.to_string(index=False))

# Use optimized policies for further analysis
print(f"\nSuccessfully optimized {len(optimized_policies)} policies:")
for policy in optimized_policies:
    print(f"  - {policy.name} ({policy.__class__.__name__})")

# Test one of the optimized policies
if optimized_policies:
    test_policy = optimized_policies[0]
    print(f"\nTesting optimized policy: {test_policy.name}")
    
    # Get action from optimized policy
    test_belief = tiger_belief
    action, run_data = test_policy.action(test_belief)
    
    print(f"Optimized policy action: {action}")
    
    # Extract planning time from info_variables list
    planning_time = 0
    for info_var in run_data.info_variables:
        if info_var.name == 'planning_time':
            planning_time = info_var.value
            break
    
    print(f"Planning time: {planning_time:.3f} seconds")

## Best Practices for Hyperparameter Optimization

Ideally, we want to perform hyper-parameter tuning, and then evaluated the chosen model afterwards. 

In [ ]:
"""
Example using the SimulationsAPI run_hyperparameter_optimization_and_evaluation method.

This method provides a simple API wrapper around the optimize_and_evaluate_planners utility function,
making it easy to use within the existing SimulationsAPI framework.
"""

from pathlib import Path
from POMDPPlanners.simulations.simulations_api import SimulationsAPI
from POMDPPlanners.utils.hyperparameter_tuning_and_eval import (
    HyperParamPlannerConfig,
    create_numerical_hyperparameter_ranges,
    get_fast_optimization_defaults
)
from POMDPPlanners.planners.mcts_planners.pomcp import POMCP
from POMDPPlanners.planners.mcts_planners.sparse_pft import SparsePFT
from POMDPPlanners.core.simulation.hyperparameter_tuning import HyperParameterOptimizationDirection
from POMDPPlanners.configs.environment_configs import EnvironmentConfigsAPI
import pandas as pd

# Setup environment and initial belief
env_configs = EnvironmentConfigsAPI(discount_factor=0.95)
environment, initial_belief = env_configs.tiger_pomdp_config(n_particles=200)

print(f"Environment: {environment.name}")
print(f"Initial belief particles: {len(initial_belief.particles)}")

# Configure multiple planners for optimization and evaluation
planner_configs = [
    # POMCP configuration
    HyperParamPlannerConfig(
        policy_cls=POMCP,
        hyper_parameters=create_numerical_hyperparameter_ranges({
            "exploration_constant": (0.1, 100.0),
            "depth": (5, 15)
        }),
        constant_parameters={
            "discount_factor": 0.95,
            "n_simulations": 500,
            "name": "OptimizedPOMCP"
        }
    ),
    
    # SparsePFT configuration  
    HyperParamPlannerConfig(
        policy_cls=SparsePFT,
        hyper_parameters=create_numerical_hyperparameter_ranges({
            "depth": (5, 15),
            "c_ucb": (0.0, 50.0),
            "beta_ucb": (0.0, 50.0)
        }),
        constant_parameters={
            "discount_factor": 0.95,
            "gamma": 0.95,
            "belief_child_num": 5,
            "n_simulations": 500,
            "name": "OptimizedSparsePFT"
        }
    )
]

print(f"\nConfigured {len(planner_configs)} planners for optimization:")
for i, config in enumerate(planner_configs):
    print(f"  {i+1}. {config.policy_cls.__name__}: {len(config.hyper_parameters)} hyperparameters")

# Set up cache directory
cache_dir = Path("./api_optimization_results")
cache_dir.mkdir(exist_ok=True)

# Get fast defaults for demonstration (use get_thorough_optimization_defaults() for production)
defaults = get_fast_optimization_defaults()
print(f"\nUsing fast optimization defaults:")
print(f"  Optimization: {defaults['n_trials']} trials, {defaults['optimization_episodes']} episodes")
print(f"  Evaluation: {defaults['evaluation_episodes']} episodes")

# Initialize SimulationsAPI and run comprehensive optimization and evaluation
print(f"\n{'='*60}")
print("STARTING API-BASED OPTIMIZATION AND EVALUATION")
print(f"{'='*60}")

api = SimulationsAPI(debug=True)

results = api.run_hyperparameter_optimization_and_evaluation(
    environment=environment,
    initial_belief=initial_belief,
    planner_configs=planner_configs,
    cache_dir=cache_dir,
    optimization_direction=HyperParameterOptimizationDirection.MAXIMIZE,
    parameter_to_optimize="average_return",
    experiment_name="api_tiger_optimization",
    # Optimization parameters (using fast defaults for demo)
    n_trials=defaults['n_trials'],
    optimization_episodes=defaults['optimization_episodes'],
    optimization_steps=defaults['optimization_steps'],
    optimization_n_jobs=-1,
    # Evaluation parameters
    evaluation_episodes=defaults['evaluation_episodes'],
    evaluation_steps=defaults['evaluation_steps'],
    evaluation_n_jobs=1,
    # Analysis parameters
    confidence_interval_level=0.95,
    alpha=0.05,
    verbose=True,
    debug=False
)

print(f"\n{'='*60}")
print("API RESULTS SUMMARY")
print(f"{'='*60}")

# Extract results
optimization_results = results['optimization_results']
evaluation_results = results['evaluation_results']
evaluation_statistics = results['evaluation_statistics']
summary = results['summary']

print(f"Successfully optimized and evaluated {summary['num_planners']} planners")
print(f"Environment: {summary['environment_name']}")
print(f"Total optimization trials: {summary['optimization_trials_per_planner']} per planner")
print(f"Total evaluation episodes: {summary['evaluation_episodes']} per planner")

# Display optimized planner details
print(f"\n🎯 OPTIMIZED PLANNERS:")
print("-" * 50)
for i, planner_info in enumerate(summary['planners']):
    print(f"{i+1}. {planner_info['policy_name']} ({planner_info['policy_type']})")
    print(f"   Best hyperparameters: {planner_info['best_hyperparameters']}")

# Display evaluation statistics summary
if not evaluation_statistics.empty:
    print(f"\n📊 EVALUATION PERFORMANCE COMPARISON:")
    print("-" * 50)
    
    # Filter for average return metric
    if 'metric' in evaluation_statistics.columns:
        avg_return_stats = evaluation_statistics[
            evaluation_statistics['metric'] == 'average_return'
        ].copy()
        
        if not avg_return_stats.empty:
            # Sort by performance (value column)
            avg_return_stats = avg_return_stats.sort_values('value', ascending=False)
            
            print(f"{'Rank':<4} {'Policy':<25} {'Avg Return':<12} {'95% CI':<20}")
            print("-" * 65)
            
            for i, (_, row) in enumerate(avg_return_stats.iterrows()):
                policy_name = row['policy']
                avg_return = row['value']
                ci_lower = row['lower_confidence_bound']
                ci_upper = row['upper_confidence_bound']
                ci_str = f"[{ci_lower:.3f}, {ci_upper:.3f}]"
                
                print(f"{i+1:<4} {policy_name:<25} {avg_return:<12.3f} {ci_str:<20}")

# Show cache directories
print(f"\n💾 RESULTS SAVED TO:")
print("-" * 30)
for cache_type, cache_path in results['cache_paths'].items():
    print(f"{cache_type}: {cache_path}")

# Access optimized policies for further use
optimized_policies = [result.policy for result in optimization_results]
print(f"\n✅ READY FOR PRODUCTION:")
print(f"   {len(optimized_policies)} optimized policies available for deployment")
print(f"   Access via results['optimization_results'][i].policy")

### Trial Configuration
- Start with fewer trials (50-100) for initial exploration
- Increase trials (200-500) for final optimization
- Use more episodes (50-100) for reliable performance estimates

### Computational Resources
- Use parallel execution (`n_jobs=-1`) when available
- Consider using distributed computing for large-scale optimization
- Monitor memory usage with large particle counts

### Evaluation Strategy
- Use consistent evaluation metrics across all configurations
- Consider multiple performance metrics (average return, success rate, planning time)
- Validate optimized policies on held-out test episodes

## Viewing Detailed Results with MLflow

The optimization and evaluation process automatically saves detailed results and metrics to MLflow tracking. To explore these results interactively:

### 1. Navigate to the cache directory
```bash
cd ./api_optimization_results
```

### 2. Launch MLflow UI
```bash
mlflow ui
```

### 3. Open browser to view results
Navigate to `http://localhost:5000` in your browser to access the MLflow UI.

### What you can explore in MLflow:

- **Hyperparameter optimization trials and metrics**: View how different parameter combinations performed during optimization
- **Policy evaluation results and statistics**: Detailed performance metrics for each optimized policy
- **Experiment comparison**: Compare results across different runs and experiments
- **Parameter importance**: Understand which hyperparameters have the most impact on performance
- **Optimization convergence plots**: Visual tracking of how optimization progressed over trials
- **Statistical confidence intervals**: Detailed statistical analysis with confidence bounds
- **Performance comparisons**: Side-by-side comparison of different policies

The MLflow UI provides an interactive dashboard that makes it easy to analyze and understand your optimization and evaluation results in depth.

## Troubleshooting Common Issues

**Low Performance After Optimization**
- Check if parameter ranges are appropriate for the problem
- Verify that the optimization direction is correct (maximize vs minimize)
- Ensure sufficient trials and episodes for reliable estimates

**Optimization Taking Too Long**
- Reduce the number of trials or episodes
- Use fewer particles in belief representation
- Decrease the planning depth or timeout limits

**Memory Issues**
- Reduce particle count in belief representation
- Use smaller planning depths
- Consider using sparse belief representations

**Convergence Problems**
- Increase the number of trials
- Adjust parameter ranges based on initial results
- Consider using different optimization algorithms in Optuna

## Summary

This notebook demonstrated:

1. **Basic hyperparameter optimization** for POMCP on Tiger POMDP
2. **Multi-environment optimization** comparing different algorithms
3. **Predefined configurations** for quick setup
4. **Results analysis** and policy comparison
5. **Best practices** for effective optimization

The optimization framework provides powerful tools for finding optimal hyperparameters across different POMDP environments and planning algorithms. For production use, consider:

- Increasing trial counts and episode numbers
- Using multiple CPU cores (`n_jobs=-1`)
- Running longer evaluations for statistical significance
- Comparing results across multiple random seeds